# 1-6. AI Agent와 LangGraph

## 학습 목표
- **ReAct**(Reasoning + Acting) 패턴의 원리를 설명할 수 있다.
- `@tool` 데코레이터로 커스텀 도구를 정의하고 LLM에 바인딩한다.
- LangGraph `create_react_agent`로 도구 호출 에이전트를 구성한다.
- `MemorySaver`를 이용해 **대화 기록이 유지되는 에이전트**를 만든다.
- Human-in-the-loop 패턴으로 에이전트의 도구 실행을 사전 승인한다.

## 핵심 키워드
`ReAct`, `Agent`, `Tool`, `LangGraph`, `create_react_agent`, `MemorySaver`, `Human-in-the-loop`

## 사전 지식
- 1-5 노트북의 **Tool Calling** 개념 (LLM이 함수 호출을 요청하는 패턴)


## 0. Agent란?

1-5에서 Tool Calling을 배웠다. LLM이 "이 함수를 이 인자로 호출해달라"고 요청하고, 우리 코드가 실행해 결과를 돌려주는 구조였다.

**Agent**는 이 루프를 **자동화**한 것이다:

```
사용자 질문 → LLM 추론 → 도구 선택 → 도구 실행 → 결과 관찰 → (반복) → 최종 답변
```

### ReAct 패턴 (Yao et al., 2022)

| 단계 | 설명 | 예시 |
|------|------|------|
| **Thought** | 현재 상황을 추론 | "환율을 알아야 달러 변환이 가능하다" |
| **Action** | 어떤 도구를 어떤 인자로 호출할지 결정 | `get_exchange_rate(base='USD', target='KRW')` |
| **Observation** | 도구 실행 결과 확인 | `1380.5` |
| 반복... | 추가 도구가 필요하면 다시 Thought → Action | |
| **Final Answer** | 충분한 정보가 모이면 최종 답변 생성 | "현재 환율 기준 약 725달러입니다" |

### LangGraph

LangGraph는 LangChain 팀이 만든 **상태 기계(state machine) 프레임워크**이다. 노드(Node)와 엣지(Edge)로 에이전트의 실행 흐름을 정의한다. `create_react_agent`는 위 ReAct 루프를 자동으로 구성해준다.

In [ ]:
import sys; sys.path.insert(0, '../..')
from common import get_chat_model, provider_badge

print(provider_badge())

## 1. 커스텀 도구 정의

`@tool` 데코레이터로 파이썬 함수를 LLM이 사용할 수 있는 도구로 변환한다. **docstring이 도구 설명**이 되므로 명확하게 작성해야 한다.

### 1.0 규정 검색 도구를 04 체인 부품으로 꽂아 넣기

`search_regulation`은 더 이상 데모용 dict 검색이 아니다. **04에서 만든 Chroma 인덱스(MMR 리트리버)를 그대로 재사용**해 진짜 전자금융감독규정을 검색한다.

- **체인의 retriever + format은 tool의 몸통**이 된다.
- **체인의 `prompt | llm` 생성 파트는 agent의 LLM이 ReAct 루프에서 대신 수행**한다 — 도구가 이미 답변을 만들면 agent LLM이 그걸 다시 요약하게 되어 이중 생성이 나기 때문이다.
- 이 구조가 바로 노트북 말미에 예고된 **Agentic RAG**(벡터 검색을 도구로 장착한 에이전트)의 가장 단순한 형태다.

> ⚠️ 이 셀을 돌리기 전에 **04 노트북을 한 번은 실행해 `./_store/efinance_rag` 인덱스가 만들어져 있어야** 한다.

In [ ]:
from pathlib import Path
from langchain_community.vectorstores import Chroma
from common import get_embeddings

CHROMA_DIR = Path('./_store/efinance_rag')
assert CHROMA_DIR.exists(), '먼저 04 노트북을 실행해 ./_store/efinance_rag 인덱스를 생성해주세요.'

regulation_vs = Chroma(
    persist_directory=str(CHROMA_DIR),
    embedding_function=get_embeddings(),
    collection_name='efinance_regulation',
)
regulation_retriever = regulation_vs.as_retriever(
    search_type='mmr',
    search_kwargs={'k': 4, 'fetch_k': 12, 'lambda_mult': 0.5},
)
print(f'✅ 규정 인덱스 로드: {regulation_vs._collection.count()}개 chunk')

In [ ]:
from langchain_core.tools import tool


@tool
def get_exchange_rate(base: str, target: str) -> float:
    """두 통화 간 현재 환율을 조회한다.

    Args:
        base: 기준 통화 코드 (예: 'USD')
        target: 대상 통화 코드 (예: 'KRW')
    """
    table = {('USD', 'KRW'): 1380.5, ('EUR', 'KRW'): 1490.2, ('JPY', 'KRW'): 9.1}
    return table.get((base.upper(), target.upper()), 0.0)


@tool
def calculate_interest(principal: float, annual_rate_pct: float, days: int) -> float:
    """원금, 연이자율(%), 기간(일)으로 단리 이자를 계산한다.

    Args:
        principal: 원금 (원)
        annual_rate_pct: 연 이자율 (%)
        days: 예치 기간 (일)
    """
    return round(principal * annual_rate_pct / 100 * days / 365, 2)


@tool
def search_regulation(query: str) -> str:
    """전자금융감독규정을 벡터 검색해 관련 조항 본문을 반환한다.
    반환된 '[제N조(...)]' 태그는 최종 답변에서 '[근거: 제N조]' 형식으로 인용할 것.

    Args:
        query: 검색어 (자연어 질의 가능 — 예: '해킹 방지대책의 주요 통제항목')
    """
    docs = regulation_retriever.invoke(query)
    if not docs:
        return '관련 규정을 찾지 못했습니다.'
    return '\n\n'.join(
        f'[{d.metadata.get("article", "알 수 없음")}] {d.page_content}'
        for d in docs
    )


tools = [get_exchange_rate, calculate_interest, search_regulation]
print(f'정의된 도구 {len(tools)}개:')
for t in tools:
    print(f'  - {t.name}: {t.description[:60]}...')

## 2. LangGraph ReAct 에이전트

`create_react_agent`는 내부적으로 다음 그래프를 구성한다:

```
          ┌──────────┐
          │  START    │
          └─────┬─────┘
                ▼
          ┌──────────┐
     ┌───▶│   LLM    │◀───┐
     │    └─────┬─────┘    │
     │          ▼          │
     │   tool_calls?       │
     │    YES ──┐  NO ──▶ END
     │          ▼          
     │   ┌──────────┐     
     └───│  TOOLS   │     
         └──────────┘     
```

LLM이 도구 호출을 요청하면 도구를 실행하고 결과를 LLM에 돌려주는 루프를 반복한다. 도구 호출이 없으면 최종 답변으로 종료한다.

In [ ]:
from langgraph.prebuilt import create_react_agent

llm = get_chat_model(temperature=0)
agent = create_react_agent(llm, tools)

print('에이전트 그래프 노드:', list(agent.get_graph().nodes.keys()))

In [ ]:
# 단일 도구 호출이 필요한 질문
result = agent.invoke({'messages': [('user', '해킹 등 방지대책으로 요구되는 주요 통제항목을 규정에서 찾아줘')]})

# 메시지 흐름 확인
for msg in result['messages']:
    role = msg.__class__.__name__
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        print(f'[{role}] tool_calls: {[(c["name"], c["args"]) for c in msg.tool_calls]}')
    elif hasattr(msg, 'name') and msg.name:
        print(f'[{role}/{msg.name}] {msg.content[:80]}')
    else:
        print(f'[{role}] {msg.content[:120]}')

In [ ]:
# 복합 질문 — 여러 도구를 순차적으로 호출해야 하는 질문
result = agent.invoke({
    'messages': [('user', '1000만원을 연 3.5%로 90일 예치하면 이자가 얼마이고, 그 이자를 달러로 환산하면?')]
})

print('=== 에이전트 실행 흐름 ===')
for msg in result['messages']:
    role = msg.__class__.__name__
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        for c in msg.tool_calls:
            print(f'  🔧 Action: {c["name"]}({c["args"]})')
    elif hasattr(msg, 'name') and msg.name:
        print(f'  👁 Observation [{msg.name}]: {msg.content}')
    elif role == 'HumanMessage':
        print(f'  ❓ Question: {msg.content[:80]}')
    else:
        print(f'  💬 Answer: {msg.content[:200]}')

## 3. 시스템 프롬프트 설정

에이전트의 행동 규칙을 시스템 프롬프트로 지정할 수 있다. 금융 도메인에서는 **규정에 없는 내용은 답변하지 않도록** 제한하는 것이 중요하다.

In [ ]:
SYSTEM_PROMPT = """당신은 금융회사의 내부 규정 담당자를 돕는 AI 어시스턴트입니다.
규칙:
1. 반드시 제공된 도구를 사용해 근거를 확인한 후 답변하세요.
2. 도구로 확인할 수 없는 질문에는 "확인할 수 없습니다"라고 답하세요.
3. 답변 마지막에 [근거: 제X조] 형식으로 출처를 명시하세요.
4. 한국어로 간결하게 답변하세요."""

agent_with_prompt = create_react_agent(llm, tools, prompt=SYSTEM_PROMPT)

# 규정에 있는 질문
result = agent_with_prompt.invoke({'messages': [('user', 'IP주소 관리대책은 어떤 내용으로 구성되어야 해?')]})
print('Q: IP주소 관리대책은 어떤 내용으로 구성되어야 해?')
print('A:', result['messages'][-1].content)
print()

# 규정에 없는 질문
result = agent_with_prompt.invoke({'messages': [('user', '내일 주가 전망 알려줘')]})
print('Q: 내일 주가 전망 알려줘')
print('A:', result['messages'][-1].content)

## 4. 대화 기억: MemorySaver

기본 에이전트는 **상태가 없다** — 매 호출마다 이전 대화를 모른다. `MemorySaver`를 체크포인터로 넘기면 `thread_id` 단위로 대화 기록이 유지된다.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()
agent_with_memory = create_react_agent(llm, tools, checkpointer=memory, prompt=SYSTEM_PROMPT)

# thread_id로 세션 구분
config = {'configurable': {'thread_id': 'session-001'}}

# 첫 번째 질문
result1 = agent_with_memory.invoke(
    {'messages': [('user', '해킹 방지대책의 주요 통제항목을 규정에서 찾아줘')]},
    config=config,
)
print('[턴 1] Q: 해킹 방지대책의 주요 통제항목을 규정에서 찾아줘')
print('[턴 1] A:', result1['messages'][-1].content)
print()

In [ ]:
# 두 번째 질문 — 이전 대화를 기억하는지 확인
result2 = agent_with_memory.invoke(
    {'messages': [('user', '그중 무선통신망 관련 항목만 좀 더 자세히 알려줘')]},
    config=config,  # 같은 thread_id
)
print('[턴 2] Q: 그중 무선통신망 관련 항목만 좀 더 자세히 알려줘')
print('[턴 2] A:', result2['messages'][-1].content)
print()

# 다른 thread_id로 호출하면 기억이 없음
config_new = {'configurable': {'thread_id': 'session-002'}}
result3 = agent_with_memory.invoke(
    {'messages': [('user', '그중 무선통신망 관련 항목만 좀 더 자세히 알려줘')]},
    config=config_new,
)
print('[새 세션] Q: 그중 무선통신망 관련 항목만 좀 더 자세히 알려줘')
print('[새 세션] A:', result3['messages'][-1].content)

## 5. 스트리밍 실행

에이전트의 사고 과정을 실시간으로 보려면 `stream`을 사용한다. 각 노드의 출력이 순차적으로 전달된다.

In [ ]:
agent_stream = create_react_agent(llm, tools, prompt=SYSTEM_PROMPT)

print('=== 스트리밍 실행 ===')
for event in agent_stream.stream(
    {'messages': [('user', '1억원을 연 4.2%로 180일 예치하면 이자가 얼마야?')]},
    stream_mode='updates',
):
    for node_name, node_output in event.items():
        print(f'\n--- 노드: {node_name} ---')
        for msg in node_output.get('messages', []):
            if hasattr(msg, 'tool_calls') and msg.tool_calls:
                for c in msg.tool_calls:
                    print(f'  🔧 {c["name"]}({c["args"]})')
            elif hasattr(msg, 'name') and msg.name:
                print(f'  👁 [{msg.name}] {msg.content}')
            else:
                print(f'  💬 {msg.content[:200]}')

## 6. 실행 제한: max_iterations

에이전트가 무한 루프에 빠지는 것을 방지하려면 **최대 반복 횟수**를 제한해야 한다. 금융 도메인에서는 비용 폭주 방지와 감사 추적을 위해 필수이다.

In [ ]:
from langgraph.prebuilt import create_react_agent

# recursion_limit: LangGraph 그래프의 최대 스텝 수 (노드 방문 횟수)
# 도구 호출 1회 = LLM 노드 + Tool 노드 = 2스텝이므로, 3번 도구 호출 허용 시 약 8 정도로 설정
agent_limited = create_react_agent(llm, tools, prompt=SYSTEM_PROMPT)

try:
    result = agent_limited.invoke(
        {'messages': [('user', '1000만원 이자 계산해줘 (연 3.5%, 90일)')]},
        config={'recursion_limit': 8},
    )
    print('정상 완료:', result['messages'][-1].content[:100])
except Exception as e:
    print(f'제한 초과: {type(e).__name__}: {e}')

## 7. Human-in-the-Loop

에이전트가 도구를 실행하기 **전에 사람의 승인**을 받는 패턴. 금융 트랜잭션이나 민감한 데이터 접근에 필수적이다.

LangGraph에서는 `interrupt_before`로 특정 노드 실행 전에 중단점을 설정할 수 있다.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

memory_hitl = MemorySaver()

# 'tools' 노드 실행 전에 중단
agent_hitl = create_react_agent(
    llm, tools,
    checkpointer=memory_hitl,
    interrupt_before=['tools'],
)

config_hitl = {'configurable': {'thread_id': 'hitl-001'}}

# 1단계: LLM이 도구 호출을 결정하면 중단됨
result = agent_hitl.invoke(
    {'messages': [('user', '이자 계산해줘: 원금 5000만원, 연 3.0%, 365일')]},
    config=config_hitl,
)

# 중단 시점의 상태 확인
last_msg = result['messages'][-1]
if hasattr(last_msg, 'tool_calls') and last_msg.tool_calls:
    print('⏸️  에이전트가 도구 실행을 요청했습니다:')
    for c in last_msg.tool_calls:
        print(f'   도구: {c["name"]}')
        print(f'   인자: {c["args"]}')
    print()
    print('→ 승인하려면 아래 셀에서 agent_hitl.invoke(None, config=config_hitl) 실행')
    print('→ 거부하려면 다른 메시지를 넣어 재시작')

In [ ]:
# 2단계: 승인 — None을 넘기면 중단된 시점부터 계속 실행
result = agent_hitl.invoke(None, config=config_hitl)
print('✅ 승인 후 최종 답변:')
print(result['messages'][-1].content)

## 8. Agent vs Chain — 언제 무엇을 쓸까?

| | **LCEL Chain** (2-4) | **Agent** (이 노트북) |
|---|---|---|
| 실행 흐름 | 고정 (retriever → prompt → LLM) | LLM이 동적으로 결정 |
| 도구 호출 | 없음 또는 1회 고정 | 0~N회 자율 결정 |
| 예측 가능성 | 높음 — 항상 같은 경로 | 낮음 — 입력에 따라 경로 변동 |
| 비용 | 예측 가능 | 변동 (도구 호출 횟수에 따라) |
| 적합한 경우 | 단순 Q&A, 고정된 RAG | 복합 질의, 다중 도구, 계획 수립 |
| 금융 규제 관점 | 감사 추적 용이 | `max_iterations` + 로깅 필수 |

### 실전 가이드
- **고정 RAG로 충분한 경우** → LCEL Chain 사용 (비용 예측 가능, 감사 용이)
- **"검색해야 할지 모르겠으면 검색하고, 계산이 필요하면 계산하라"** → Agent 사용
- **폐쇄망에서는** Agent의 도구 호출 횟수 제한(`recursion_limit`)과 실행 로그를 반드시 남겨야 함

Day 2에서는 이 기반 위에 **Agentic RAG**(벡터 검색을 도구로 장착한 에이전트)와 **Adaptive RAG**(쿼리 복잡도에 따라 경로를 분기하는 에이전트)를 배운다.


## 더 읽어보기
- Yao et al., [ReAct: Synergizing Reasoning and Acting in Language Models (2022)](https://arxiv.org/abs/2210.03629)
- [LangGraph 공식 문서](https://langchain-ai.github.io/langgraph/)
- [LangGraph — create_react_agent](https://langchain-ai.github.io/langgraph/reference/prebuilt/#create_react_agent)